# PySpark inner join

## Source data

The source comes from jupyter-pyspark/f1-sourcefiles

# Inner the data from the results.csv and races.csv 


# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/04/25 23:06:54 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/25 23:06:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-22a98cd8-8ed9-4d5a-8d9c-d4939acd987c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 278ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

# Load results.csv and races.csv into dataframes

In [2]:
# Load csv files into dataframes
#  Contains headers
# Infers schema

results = spark.read.csv("f1-sourcefiles/results.csv", header=True, inferSchema=True)

races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True)


26/04/25 23:07:08 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# Show the data type of the dataframes


In [3]:
# results 

results.printSchema()

# races

races.printSchema()

root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: string (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: string (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: string (nullable = true)
 |-- fastestLap: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- fastestLapTime: string (nullable = true)
 |-- fastestLapSpeed: string (nullable = true)
 |-- statusId: integer (nullable = true)

root
 |-- raceId: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- round: integer (nullable = true)
 |-- circuitId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- date: date (nullable = true)
 |-- time: string (nulla

# Use Alias in an INNER JOIN

In [7]:
# Inner join on same column name i.e. raceid


# Create and alias for a Dataframe just like tabel alias in SQL

results = results.alias("rslt")

races = races.alias("rcs")

# join on raceid and look at shuffle size to optimise on?


joined_df = results.join(races, col("rslt.raceId") == col("rcs.raceId"), how="inner") \
    .filter("rcs.year == 2024") \
    .select(col("rslt.resultId"), col("rcs.raceId"), col("rcs.year"), col("rcs.round"), col("rcs.name").alias("race_name"))

joined_df.explain()

joined_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [resultId#17, raceId#70, year#71, round#72, name#74 AS race_name#390]
   +- BroadcastHashJoin [raceId#18], [raceId#70], Inner, BuildRight, false
      :- Filter isnotnull(raceId#18)
      :  +- FileScan csv [resultId#17,raceId#18] Batched: false, DataFilters: [isnotnull(raceId#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-source..., PartitionFilters: [], PushedFilters: [IsNotNull(raceId)], ReadSchema: struct<resultId:int,raceId:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=272]
         +- Filter ((isnotnull(year#71) AND (year#71 = 2024)) AND isnotnull(raceId#70))
            +- FileScan csv [raceId#70,year#71,round#72,name#74] Batched: false, DataFilters: [isnotnull(year#71), (year#71 = 2024), isnotnull(raceId#70)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/ho

# Force a broadcast hash join INNER JOIN 

In [6]:
# Inner join on same column name i.e. raceid

from pyspark.sql.functions import broadcast

# Create and alias for a Dataframe just like tabel alias in SQL

results = results.alias("rslt")

races = races.alias("rcs")

# join on raceid and look at shuffle size to optimise on?

# Force a broadcast join
joined_df = results.join(broadcast(races), col("rslt.raceId") == col("rcs.raceId"), how="inner") \
    .filter("rcs.year == 2024") \
    .select(col("rslt.resultId"), col("rcs.raceId"), col("rcs.year"), col("rcs.round"), col("rcs.name").alias("race_name"))

joined_df.explain()

joined_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [resultId#17, raceId#70, year#71, round#72, name#74 AS race_name#285]
   +- BroadcastHashJoin [raceId#18], [raceId#70], Inner, BuildRight, false
      :- Filter isnotnull(raceId#18)
      :  +- FileScan csv [resultId#17,raceId#18] Batched: false, DataFilters: [isnotnull(raceId#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-source..., PartitionFilters: [], PushedFilters: [IsNotNull(raceId)], ReadSchema: struct<resultId:int,raceId:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=168]
         +- Filter ((isnotnull(year#71) AND (year#71 = 2024)) AND isnotnull(raceId#70))
            +- FileScan csv [raceId#70,year#71,round#72,name#74] Batched: false, DataFilters: [isnotnull(year#71), (year#71 = 2024), isnotnull(raceId#70)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/ho

# This is a sample showing a broadcast join is used.

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [resultId#17, raceId#70, year#71, round#72, name#74 AS race_name#285]
   +- BroadcastHashJoin [raceId#18], [raceId#70], Inner, BuildRight, false
      :- Filter isnotnull(raceId#18)


# How to get the current broadcast  threshold


In [9]:
# Safe way — use SparkContext's byteString parser
threshold_bytes = spark._jvm.org.apache.spark.network.util.JavaUtils.byteStringAsBytes(
    spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
)
print(f"Broadcast threshold: {threshold_bytes / (1024 * 1024):.1f} MB")

Broadcast threshold: 10.0 MB


# Set lower Broadcast hash join threshold to makit not broadcast join
